In [1]:
!pip install requests beautifulsoup4 pandas

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://quotes.toscrape.com"

response = requests.get(url)

print(response.status_code)
print(response.text[:500])

200
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div cla


In [3]:
# kap_access_check.py

import time
import requests
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from urllib.parse import urljoin

BASE_URL = "https://www.kap.org.tr"
TEST_PATHS = [
    "/tr",
    "/tr/bildirim-sorgu",
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; academic-research-bot/0.1; "
        "+https://github.com/yourusername/bist-insight-assistant)"
    ),
    "Accept-Language": "tr-TR,tr;q=0.9,en;q=0.8",
}


def check_robots():
    robots_url = urljoin(BASE_URL, "/robots.txt")
    rp = RobotFileParser()
    rp.set_url(robots_url)

    try:
        rp.read()
        print(f"[OK] robots.txt okundu: {robots_url}")
    except Exception as e:
        print(f"[WARN] robots.txt okunamadı: {e}")
        return

    for path in TEST_PATHS:
        allowed = rp.can_fetch(HEADERS["User-Agent"], urljoin(BASE_URL, path))
        print(f"[ROBOTS] {path} -> {'Allowed' if allowed else 'Disallowed'}")


def fetch_page(path):
    url = urljoin(BASE_URL, path)
    print(f"\n[TEST] {url}")

    try:
        start = time.time()
        response = requests.get(url, headers=HEADERS, timeout=20)
        elapsed = time.time() - start
    except requests.RequestException as e:
        print(f"[FAIL] Request hatası: {e}")
        return None

    print(f"Status code: {response.status_code}")
    print(f"Response time: {elapsed:.2f}s")
    print(f"Content-Type: {response.headers.get('content-type')}")
    print(f"Content length: {len(response.text):,} chars")

    suspicious_terms = ["captcha", "access denied", "forbidden", "bot", "blocked"]
    lower_text = response.text.lower()

    found_flags = [term for term in suspicious_terms if term in lower_text]
    if found_flags:
        print(f"[WARN] Olası blok/captcha sinyali: {found_flags}")

    if response.status_code == 200:
        print("[OK] Sayfa erişilebilir.")
    elif response.status_code in [403, 429]:
        print("[RISK] Blok / rate limit ihtimali var.")
    else:
        print("[WARN] Beklenmeyen status code.")

    return response


def inspect_html(response):
    if response is None:
        return

    soup = BeautifulSoup(response.text, "html.parser")

    title = soup.title.get_text(strip=True) if soup.title else None
    print(f"Page title: {title}")

    text = soup.get_text(" ", strip=True)
    keywords = ["Bildirim", "Şirket", "Finansal", "Açıklama", "KAP"]

    matches = [kw for kw in keywords if kw.lower() in text.lower()]
    print(f"Bulunan anahtar kelimeler: {matches}")

    links = soup.find_all("a")
    print(f"Link sayısı: {len(links)}")

    sample_links = []
    for a in links[:10]:
        href = a.get("href")
        label = a.get_text(" ", strip=True)
        if href:
            sample_links.append((label[:50], href))

    print("\nÖrnek linkler:")
    for label, href in sample_links:
        print(f"- {label} -> {href}")


def main():
    print("=" * 60)
    print("KAP ACCESS CHECK")
    print("=" * 60)

    check_robots()

    for path in TEST_PATHS:
        response = fetch_page(path)
        inspect_html(response)
        time.sleep(3)  # nazik bekleme

    print("\n" + "=" * 60)
    print("SONUÇ YORUMU")
    print("=" * 60)
    print("""
Eğer:
- robots.txt ilgili path için Allowed diyorsa,
- status code 200 ise,
- captcha / blocked / 403 / 429 yoksa,
- HTML içinde bildirim/sorgu içeriği görünüyorsa,

düşük frekanslı, cache'li, araştırma amaçlı scraping teknik olarak mümkün görünüyor.

Ama:
- 403 / 429 alırsan,
- captcha çıkarsa,
- robots.txt Disallowed diyorsa,

scraping'i bırakıp alternatif veri kaynağı veya resmi API/abonelik yoluna bakmak daha doğru olur.
""")


if __name__ == "__main__":
    main()

KAP ACCESS CHECK
[OK] robots.txt okundu: https://www.kap.org.tr/robots.txt
[ROBOTS] /tr -> Disallowed
[ROBOTS] /tr/bildirim-sorgu -> Disallowed

[TEST] https://www.kap.org.tr/tr
Status code: 200
Response time: 0.18s
Content-Type: text/html; charset=utf-8
Content length: 164,894 chars
[WARN] Olası blok/captcha sinyali: ['bot', 'blocked']
[OK] Sayfa erişilebilir.
Page title: KAP
Bulunan anahtar kelimeler: ['Bildirim', 'Şirket', 'Finansal', 'Açıklama', 'KAP']
Link sayısı: 47

Örnek linkler:
-  -> #
- Bildirim İçeriklerinde ve Eklerinde Ara -> /tr/search//1
- Tüm Duyurular -> /tr/about/mevzuat-duyurular-ve-kilavuzlar/tab-content/duyurular
- Tümü -> /about/mevzuat-duyurular-ve-kilavuzlar/tab-content/duyurular
- Tümünü Gör -> /tr/about/mevzuat-duyurular-ve-kilavuzlar/tab-content/mevzuatlar_ve_kilavuzlar
- KAP Yönergesi -> /tr/api/about/content-file/402881cf91d6efd60191d6f08e900001
- Sıkça Sorulan Sorular -> /tr/api/about/content-file/402881cf91d6efd60191d6f04e530000
- Veri Analiz Platformu -

In [4]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

# KAP API endpoint
URL = "https://www.kap.org.tr/tr/api/disclosure/members/byCriteria"

HEADERS = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0"
}

BIST30_TICKERS = [
    "AKBNK", "ARCLK", "ASELS", "BIMAS", "CIMSA",
    "EKGYO", "EREGL", "FROTO", "GARAN", "GUBRF",
    "HALKB", "ISCTR", "KCHOL", "KONTR", "KRDMD",
    "MGROS", "ODAS", "PETKM", "PGSUS", "SAHOL",
    "SASA", "SISE", "TAVHL", "TCELL", "THYAO",
    "TKFEN", "TOASO", "TUPRS", "VAKBN", "YKBNK"
]

def fetch_disclosures(from_date: str, to_date: str) -> list:
    """
    Fetch all disclosures for a given date range.
    Dates in format: YYYY-MM-DD
    """
    payload = {
        "fromDate": from_date,
        "toDate": to_date,
        "memberType": "IGS",
        "mkkMemberOidList": [],
        "bdkMemberOidList": [],
        "disclosureIndexList": [],
        "subjectList": [],
        "bdkReview": "",
        "disclosureClass": "",
        "fromSrc": False,
        "inactiveMkkMemberOidList": [],
        "index": "",
        "isLate": "",
        "mainSector": "",
        "marketOid": "",
        "period": "",
        "ruleType": "",
        "sector": "",
        "srcCategory": "",
        "subSector": "",
        "term": "",
        "year": ""
    }
    
    response = requests.post(URL, json=payload, headers=HEADERS)
    response.raise_for_status()
    return response.json()

# Test: bugünün verisi
data = fetch_disclosures("2026-04-30", "2026-04-30")
print(f"Bugün: {len(data)} duyuru")
print(data[0])

Bugün: 66 duyuru
{'publishDate': '30.04.2026 10:56:31', 'fundCode': None, 'kapTitle': 'KORTEKS MENSUCAT SANAYİ VE TİCARET A.Ş.', 'isOldKap': False, 'disclosureClass': 'ODA', 'disclosureType': 'CA', 'disclosureCategory': 'STT', 'summary': 'TRSKORT22718 ISIN kodlu Korteks ihracının 1. kupon ödeme dönemine ilişkin açıklama', 'subject': 'Pay Dışında Sermaye Piyasası Aracı İşlemlerine İlişkin Bildirim (Faiz İçeren)', 'relatedStocks': None, 'year': None, 'ruleType': '-', 'period': None, 'disclosureIndex': 1599200, 'isLate': False, 'stockCodes': 'KORTS', 'hasMultiLanguageSupport': False, 'attachmentCount': 0, 'modifyStatus': None}


In [5]:
from pathlib import Path

def fetch_date_range(start_date: str, end_date: str, delay: float = 1.0) -> list:
    """
    Fetch disclosures month by month to avoid large responses.
    """
    all_records = []
    
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    while current <= end:
        # Her ay için ayrı istek
        month_end = (current.replace(day=1) + timedelta(days=32)).replace(day=1) - timedelta(days=1)
        month_end = min(month_end, end)
        
        from_str = current.strftime("%Y-%m-%d")
        to_str = month_end.strftime("%Y-%m-%d")
        
        try:
            records = fetch_disclosures(from_str, to_str)
            all_records.extend(records)
            print(f"✅ {from_str} -> {to_str}: {len(records)} duyuru")
        except Exception as e:
            print(f"❌ {from_str} -> {to_str}: {e}")
        
        time.sleep(delay)  # sunucuya saygılı ol
        current = month_end + timedelta(days=1)
    
    return all_records

# Son 12 ay çek
end_date = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")

print(f"Çekiliyor: {start_date} -> {end_date}")
raw_records = fetch_date_range(start_date, end_date)
print(f"\nToplam: {len(raw_records)} duyuru")

Çekiliyor: 2025-04-30 -> 2026-04-30
✅ 2025-04-30 -> 2025-04-30: 804 duyuru
✅ 2025-05-01 -> 2025-05-31: 2000 duyuru
✅ 2025-06-01 -> 2025-06-30: 2000 duyuru
✅ 2025-07-01 -> 2025-07-31: 2000 duyuru
✅ 2025-08-01 -> 2025-08-31: 2000 duyuru
✅ 2025-09-01 -> 2025-09-30: 2000 duyuru
✅ 2025-10-01 -> 2025-10-31: 2000 duyuru
✅ 2025-11-01 -> 2025-11-30: 2000 duyuru
✅ 2025-12-01 -> 2025-12-31: 2000 duyuru
✅ 2026-01-01 -> 2026-01-31: 2000 duyuru
✅ 2026-02-01 -> 2026-02-28: 2000 duyuru
✅ 2026-03-01 -> 2026-03-31: 2000 duyuru
✅ 2026-04-01 -> 2026-04-30: 2000 duyuru

Toplam: 24804 duyuru


In [6]:
data = fetch_disclosures("2026-04-30", "2026-04-30")
print(data[0].keys())   # hangi alanlar geliyor?
print(data[0])          # tam bir kayıt nasıl görünüyor?

dict_keys(['publishDate', 'fundCode', 'kapTitle', 'isOldKap', 'disclosureClass', 'disclosureType', 'disclosureCategory', 'summary', 'subject', 'relatedStocks', 'year', 'ruleType', 'period', 'disclosureIndex', 'isLate', 'stockCodes', 'hasMultiLanguageSupport', 'attachmentCount', 'modifyStatus'])
{'publishDate': '30.04.2026 11:14:43', 'fundCode': None, 'kapTitle': 'İSTANBUL TAKAS VE SAKLAMA BANKASI A.Ş.', 'isOldKap': False, 'disclosureClass': 'DKB', 'disclosureType': 'DUY', 'disclosureCategory': 'ODA', 'summary': 'İşleme Açılan Temerrüt Sırası', 'subject': 'Temerrüt İşlemi', 'relatedStocks': 'KONTR', 'year': None, 'ruleType': '-', 'period': None, 'disclosureIndex': 1599204, 'isLate': False, 'stockCodes': None, 'hasMultiLanguageSupport': True, 'attachmentCount': 0, 'modifyStatus': None}


In [8]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
from pathlib import Path

URL = "https://www.kap.org.tr/tr/api/disclosure/members/byCriteria"

HEADERS = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (Academic research project)"
}

BIST30_TICKERS = [
    "AKBNK", "ARCLK", "ASELS", "BIMAS", "CIMSA",
    "EKGYO", "EREGL", "FROTO", "GARAN", "GUBRF",
    "HALKB", "ISCTR", "KCHOL", "KONTR", "KRDMD",
    "MGROS", "ODAS", "PETKM", "PGSUS", "SAHOL",
    "SASA", "SISE", "TAVHL", "TCELL", "THYAO",
    "TKFEN", "TOASO", "TUPRS", "VAKBN", "YKBNK"
]

def fetch_disclosures(from_date: str, to_date: str) -> list:
    payload = {
        "fromDate": from_date,
        "toDate": to_date,
        "memberType": "IGS",
        "mkkMemberOidList": [],
        "bdkMemberOidList": [],
        "disclosureIndexList": [],
        "subjectList": [],
        "bdkReview": "",
        "disclosureClass": "",
        "fromSrc": False,
        "inactiveMkkMemberOidList": [],
        "index": "",
        "isLate": "",
        "mainSector": "",
        "marketOid": "",
        "period": "",
        "ruleType": "",
        "sector": "",
        "srcCategory": "",
        "subSector": "",
        "term": "",
        "year": ""
    }

    response = requests.post(URL, json=payload, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()


def parse_disclosures(raw: list) -> pd.DataFrame:
    if not raw:
        return pd.DataFrame()

    records = []
    for item in raw:
        # relatedStocks: "THYAO" veya "THYAO, GARAN" veya None
        related = item.get("relatedStocks") or ""
        tickers = [t.strip() for t in related.split(",") if t.strip()]

        # Her hisse için ayrı satır (explode yerine burada halledelim)
        for ticker in tickers if tickers else [""]:
            records.append({
                "ticker":             ticker.upper(),
                "disclosure_index":   item.get("disclosureIndex"),
                "published_at":       item.get("publishDate"),        # "30.04.2026 11:14:43"
                "kap_title":          item.get("kapTitle"),           # Şirketi bildiren kurum
                "subject":            item.get("subject"),
                "summary":            item.get("summary"),
                "disclosure_class":   item.get("disclosureClass"),
                "disclosure_type":    item.get("disclosureType"),
                "disclosure_category": item.get("disclosureCategory"),
                "rule_type":          item.get("ruleType"),
                "year":               item.get("year"),
                "period":             item.get("period"),
                "is_late":            item.get("isLate", False),
                "is_old_kap":         item.get("isOldKap", False),
                "has_multi_language": item.get("hasMultiLanguageSupport", False),
                "attachment_count":   item.get("attachmentCount", 0),
                "url": f"https://www.kap.org.tr/tr/Bildirim/{item.get('disclosureIndex', '')}",
            })

    df = pd.DataFrame(records)

    # Tarih parse — KAP formatı: "30.04.2026 11:14:43"
    df["published_at"] = pd.to_datetime(
        df["published_at"], format="%d.%m.%Y %H:%M:%S", errors="coerce"
    )

    return df


def filter_bist30(df: pd.DataFrame) -> pd.DataFrame:
    """Sadece BIST30 hisselerini tut."""
    return df[df["ticker"].isin(BIST30_TICKERS)].copy()


def fetch_date_range(start_date: str, end_date: str, delay: float = 1.5) -> pd.DataFrame:
    all_dfs = []

    current = datetime.strptime(start_date, "%Y-%m-%d")
    end    = datetime.strptime(end_date,   "%Y-%m-%d")

    while current <= end:
        month_end = (current.replace(day=1) + timedelta(days=32)).replace(day=1) - timedelta(days=1)
        month_end = min(month_end, end)

        from_str = current.strftime("%Y-%m-%d")
        to_str   = month_end.strftime("%Y-%m-%d")

        try:
            raw     = fetch_disclosures(from_str, to_str)
            df_all  = parse_disclosures(raw)
            df_bist = filter_bist30(df_all)

            all_dfs.append(df_bist)
            print(f"✅ {from_str} → {to_str}: {len(raw)} toplam | {len(df_bist)} BIST30")

        except requests.HTTPError as e:
            print(f"❌ HTTP {e.response.status_code} — {from_str} → {to_str}")
        except Exception as e:
            print(f"❌ {from_str} → {to_str}: {e}")

        time.sleep(delay)
        current = month_end + timedelta(days=1)

    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()


# --- Çalıştır ---
if __name__ == "__main__":
    end_date   = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")

    print(f"Çekiliyor: {start_date} → {end_date}\n")
    df = fetch_date_range(start_date, end_date)

    print(f"\nToplam BIST30 duyurusu: {len(df)}")
    print(df.dtypes)
    print(df.head(3))

    # Kaydet
    out = Path("data/raw/kap")
    out.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out / "kap_disclosures.parquet", index=False)
    print(f"\n💾 Kaydedildi → {out / 'kap_disclosures.parquet'}")

Çekiliyor: 2025-04-30 → 2026-04-30

✅ 2025-04-30 → 2025-04-30: 804 toplam | 10 BIST30
✅ 2025-05-01 → 2025-05-31: 2000 toplam | 29 BIST30
✅ 2025-06-01 → 2025-06-30: 2000 toplam | 48 BIST30
✅ 2025-07-01 → 2025-07-31: 2000 toplam | 26 BIST30
✅ 2025-08-01 → 2025-08-31: 2000 toplam | 23 BIST30
✅ 2025-09-01 → 2025-09-30: 2000 toplam | 52 BIST30
✅ 2025-10-01 → 2025-10-31: 2000 toplam | 23 BIST30
✅ 2025-11-01 → 2025-11-30: 2000 toplam | 33 BIST30
✅ 2025-12-01 → 2025-12-31: 2000 toplam | 66 BIST30
✅ 2026-01-01 → 2026-01-31: 2000 toplam | 59 BIST30
✅ 2026-02-01 → 2026-02-28: 2000 toplam | 21 BIST30
✅ 2026-03-01 → 2026-03-31: 2000 toplam | 38 BIST30
✅ 2026-04-01 → 2026-04-30: 2000 toplam | 23 BIST30

Toplam BIST30 duyurusu: 451
ticker                            str
disclosure_index                int64
published_at           datetime64[us]
kap_title                         str
subject                           str
summary                           str
disclosure_class                  str
disclos

In [9]:
raw  = fetch_disclosures("2026-04-01", "2026-04-30")
df   = parse_disclosures(raw)
bist = filter_bist30(df)

print(f"Toplam: {len(df)} kayıt | BIST30: {len(bist)} kayıt")
print(bist["ticker"].value_counts().head(10))   # hangi hisse kaç duyuru?
print(bist.head(3))

Toplam: 2306 kayıt | BIST30: 23 kayıt
ticker
KONTR    11
SASA      2
EKGYO     2
EREGL     2
TAVHL     1
TKFEN     1
SAHOL     1
SISE      1
MGROS     1
TUPRS     1
Name: count, dtype: int64
   ticker  disclosure_index        published_at  \
0   KONTR           1599204 2026-04-30 11:14:43   
10  KONTR           1599190 2026-04-30 10:40:03   
67   SASA           1599106 2026-04-30 09:03:15   

                                 kap_title  \
0   İSTANBUL TAKAS VE SAKLAMA BANKASI A.Ş.   
10                     BORSA İSTANBUL A.Ş.   
67             KAMUYU AYDINLATMA PLATFORMU   

                                             subject  \
0                                    Temerrüt İşlemi   
10  BISTECH Pay Piyasası Alım Satım Sistemi Duyurusu   
67                          Pay Alım Satım Bildirimi   

                          summary disclosure_class disclosure_type  \
0   İşleme Açılan Temerrüt Sırası              DKB             DUY   
10       Toptan Alış Satış İşlemi              DKB    

In [10]:
# BIST30'a giren ama relatedStocks'ta yakalanmayanları bul
sample = raw[:50]
for item in sample:
    title = item.get("kapTitle", "")
    related = item.get("relatedStocks", "")
    # Thyao, Garan vb. içeren kapTitle'lar var mı?
    if any(t in title.upper() for t in BIST30_TICKERS):
        print(f"kapTitle: {title[:60]} | related: {related}")

kapTitle: GARANTİ YATIRIM ORTAKLIĞI A.Ş. | related: None
kapTitle: ASELSAN ELEKTRONİK SANAYİ VE TİCARET A.Ş. | related: None


In [18]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

URL = "https://www.kap.org.tr/tr/api/disclosure/members/byCriteria"

HEADERS = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
}

BIST30 = {
    "AKBNK", "ARCLK", "ASELS", "BIMAS", "CIMSA",
    "EKGYO", "EREGL", "FROTO", "GARAN", "GUBRF",
    "HALKB", "ISCTR", "KCHOL", "KONTR", "KRDMD",
    "MGROS", "ODAS",  "PETKM", "PGSUS", "SAHOL",
    "SASA",  "SISE",  "TAVHL", "TCELL", "THYAO",
    "TKFEN", "TOASO", "TUPRS", "VAKBN", "YKBNK",
}

def kap_cek(baslangic, bitis):
    payload = {
        "fromDate": baslangic,
        "toDate": bitis,
        "memberType": "IGS",
        "mkkMemberOidList": [], "bdkMemberOidList": [],
        "disclosureIndexList": [], "subjectList": [],
        "bdkReview": "", "disclosureClass": "", "fromSrc": False,
        "inactiveMkkMemberOidList": [], "index": "", "isLate": "",
        "mainSector": "", "marketOid": "", "period": "",
        "ruleType": "", "sector": "", "srcCategory": "",
        "subSector": "", "term": "", "year": "",
    }
    yanit = requests.post(URL, json=payload, headers=HEADERS, timeout=30)
    yanit.raise_for_status()
    return yanit.json()

def ticker_bul(item):
    # Her iki alandan da ticker topla
    related = item.get("relatedStocks") or ""
    stock_codes = item.get("stockCodes") or ""
    
    ham = related + "," + stock_codes
    tickers = [t.strip().upper() for t in ham.split(",") if t.strip()]
    return list(set(t for t in tickers if t in BIST30))

def isle(ham_liste):
    satirlar = []
    for item in ham_liste:
        tickers = ticker_bul(item)
        for ticker in tickers:
            satirlar.append({
                "ticker":      ticker,
                "tarih":       item.get("publishDate"),
                "konu":        item.get("subject"),
                "ozet":        item.get("summary"),
                "sinif":       item.get("disclosureClass"),
                "link":        f"https://www.kap.org.tr/tr/Bildirim/{item.get('disclosureIndex')}",
            })
    df = pd.DataFrame(satirlar)
    if not df.empty:
        df["tarih"] = pd.to_datetime(df["tarih"], format="%d.%m.%Y %H:%M:%S", errors="coerce")
    return df

# ── Çalıştır ──────────────────────────────────────────────────────────────

bitis     = datetime.today()
baslangic = bitis - timedelta(days=365)

tum_df = []
current = baslangic

while current <= bitis:
    ay_sonu = current + timedelta(days=6) 
    ay_sonu = min(ay_sonu, bitis)

    from_str = current.strftime("%Y-%m-%d")
    to_str   = ay_sonu.strftime("%Y-%m-%d")

    try:
        ham  = kap_cek(from_str, to_str)
        df   = isle(ham)
        tum_df.append(df)
        print(f"✅ {from_str} → {to_str}: {len(ham)} toplam, {len(df)} BIST30")
    except Exception as e:
        print(f"❌ {from_str} → {to_str}: {e}")

    time.sleep(1.5)
    current = ay_sonu + timedelta(days=1)

sonuc = pd.concat(tum_df, ignore_index=True)
print(f"\nToplam: {len(sonuc)} duyuru")
print(sonuc["ticker"].value_counts().head(10))

sonuc.to_csv("kap_duyurular.csv", index=False)
print("\n✅ kap_duyurular.csv olarak kaydedildi")

✅ 2025-04-30 → 2025-05-06: 1776 toplam, 121 BIST30
✅ 2025-05-07 → 2025-05-13: 2000 toplam, 134 BIST30
✅ 2025-05-14 → 2025-05-20: 1117 toplam, 100 BIST30
✅ 2025-05-21 → 2025-05-27: 1470 toplam, 72 BIST30
✅ 2025-05-28 → 2025-06-03: 1587 toplam, 131 BIST30
✅ 2025-06-04 → 2025-06-10: 512 toplam, 49 BIST30
✅ 2025-06-11 → 2025-06-17: 1113 toplam, 73 BIST30
✅ 2025-06-18 → 2025-06-24: 1147 toplam, 97 BIST30
✅ 2025-06-25 → 2025-07-01: 1283 toplam, 92 BIST30
✅ 2025-07-02 → 2025-07-08: 1072 toplam, 70 BIST30
✅ 2025-07-09 → 2025-07-15: 779 toplam, 30 BIST30
✅ 2025-07-16 → 2025-07-22: 950 toplam, 40 BIST30
✅ 2025-07-23 → 2025-07-29: 1082 toplam, 80 BIST30
✅ 2025-07-30 → 2025-08-05: 1280 toplam, 137 BIST30
✅ 2025-08-06 → 2025-08-12: 2000 toplam, 87 BIST30
✅ 2025-08-13 → 2025-08-19: 2000 toplam, 75 BIST30
✅ 2025-08-20 → 2025-08-26: 1101 toplam, 49 BIST30
✅ 2025-08-27 → 2025-09-02: 1544 toplam, 74 BIST30
✅ 2025-09-03 → 2025-09-09: 997 toplam, 73 BIST30
✅ 2025-09-10 → 2025-09-16: 1052 toplam, 60 BIST30

In [19]:
print(f"Toplam satır: {len(sonuc)}")
print(f"Unique ticker: {sonuc['ticker'].nunique()}")
print("\nTicker dağılımı:")
print(sonuc["ticker"].value_counts())

Toplam satır: 4973
Unique ticker: 30

Ticker dağılımı:
ticker
EKGYO    542
YKBNK    406
VAKBN    359
AKBNK    323
GARAN    285
KONTR    278
ISCTR    275
HALKB    209
SISE     166
ARCLK    143
SASA     134
TKFEN    124
MGROS    120
BIMAS    119
TUPRS    115
KCHOL    111
TCELL    110
THYAO    108
EREGL    106
GUBRF    106
TAVHL    106
ASELS     98
FROTO     96
TOASO     88
PGSUS     85
SAHOL     85
KRDMD     79
PETKM     72
ODAS      63
CIMSA     62
Name: count, dtype: int64


In [20]:
out_dir = Path("../data/raw/kap")
out_dir.mkdir(parents=True, exist_ok=True)

sonuc.to_parquet(out_dir / "announcements.parquet", index=False)
print(f"✅ {len(sonuc)} duyuru kaydedildi")
print(f"   Tarih: {sonuc['tarih'].min()} → {sonuc['tarih'].max()}")

✅ 4973 duyuru kaydedildi
   Tarih: 2025-04-30 00:00:53 → 2026-04-30 11:14:43


In [21]:
sonuc.sample(10)

,ticker,tarih,konu,ozet,sinif,link
4690,SASA,2026-04-10 18:17:18,Bağımsız Denetim Kuruluşunun Belirlenmesi,Bağımsız Denetim Kuruluşu Seçimi,ODA,https://www.kap.org.tr/tr/Bildirim/1590969
3339,EKGYO,2026-01-16 20:12:48,Özel Durum Açıklaması (Genel),İstanbul Beşiktaş Dikilitaş Güney İhalesi 2.Ot...,ODA,https://www.kap.org.tr/tr/Bildirim/1544363
4301,TCELL,2026-03-11 23:24:49,Sürdürülebilirlik Uyum Raporu,2025 Sürdürülebilirlik İlkeleri Uyum Raporu,DG,https://www.kap.org.tr/tr/Bildirim/1571310
936,ASELS,2025-07-02 09:49:28,Yeni İş İlişkisi,Sözleşme İmzalanması,ODA,https://www.kap.org.tr/tr/Bildirim/1454020
4636,CIMSA,2026-04-01 09:43:41,Özel Durum Açıklaması (Genel),Yönetim Kurulu Komite Üyeliklerinin Belirlenmesi,ODA,https://www.kap.org.tr/tr/Bildirim/1582016
3665,YKBNK,2026-02-05 08:06:07,Geleceğe Dönük Değerlendirmeler,Geleceğe Yönelik Değerlendirmeler,ODA,https://www.kap.org.tr/tr/Bildirim/1552726
1439,MGROS,2025-09-02 16:12:40,Pay Bazında Devre Kesici Bildirimi,MGROS.E İşlem Sırasında Devre Kesici Uygulamas...,DKB,https://www.kap.org.tr/tr/Bildirim/1485391
776,EREGL,2025-06-18 08:46:00,Pay Alım Satım Bildirimi,Pay Alım Satım Bildirimi\n,DKB,https://www.kap.org.tr/tr/Bildirim/1449600
2760,AKBNK,2025-12-10 17:38:08,Pay Dışında Sermaye Piyasası Aracı İşlemlerine...,Yurtdışı Piyasalara Tahvil İhracı,ODA,https://www.kap.org.tr/tr/Bildirim/1525665
1945,ASELS,2025-10-21 18:13:11,SPK İşlem Yasağı Nedeniyle Pay Duyurusu,SPK İşlem Yasağı Nedeniyle Pay Duyurusu,DKB,https://www.kap.org.tr/tr/Bildirim/1506510


In [25]:
import requests
from bs4 import BeautifulSoup
import requests
url = "https://www.kap.org.tr/tr/Bildirim/1590969"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers, timeout=15, verify=False)
soup = BeautifulSoup(r.text, "html.parser")
print(r.status_code)
print(soup.get_text()[:500])

200
KAPENBildirim SorgularıBildirim SorgularıBugün Gelen BildirimlerBeklenen BildirimlerDetaylı SorgulamaFinansal Tablo Kalem SorgulamaŞirketlerŞirketlerBIST ŞirketleriYatırım KuruluşlarıPortföy Yönetim ŞirketleriBağımsız Denetim KuruluşlarıTüm ŞirketlerDiğer KAP Üyeleri ve İşlem Görmeyen ŞirketlerDerecelendirme ŞirketleriKripto Varlık Hizmet SağlayıcıKAP Üyeliği Sona Eren ŞirketlerFonlarFonlarBorsa Yatırım FonlarıYatırım FonlarıEmeklilik Yatırım FonlarıOKS Emeklilik Yatırım FonlarıYabancı Yatırım F


/Users/ulasmerttoy/Desktop/bist30/bist-30-assistant/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kap.org.tr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings


In [26]:
# Bildirim detay API'si var mı?
url = "https://www.kap.org.tr/tr/api/disclosure/1590969"
r = requests.get(url, headers=headers, timeout=15, verify=False)
print(r.status_code)
print(r.text[:500])

404
<!DOCTYPE html><html id="__next_error__"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="script" fetchPriority="low" href="/_next/static/chunks/webpack-68d5eb719ea105e3.js"/><script src="/_next/static/chunks/fd9d1056-1374b3982f0f1d4c.js" async=""></script><script src="/_next/static/chunks/2117-aea548c9137939ae.js" async=""></script><script src="/_next/static/chunks/main-app-5294d1645fe5526e.js" async=""></script><meta name


/Users/ulasmerttoy/Desktop/bist30/bist-30-assistant/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kap.org.tr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings


In [1]:
from playwright.async_api import async_playwright
import asyncio

async def kap_metin_cek(bildirim_index: int) -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(f"https://www.kap.org.tr/tr/Bildirim/{bildirim_index}")
        await page.wait_for_timeout(2000)  # JS yüklensin
        
        # Ana içerik alanını bul
        icerik = await page.inner_text("body")
        await browser.close()
        return icerik

# Test: SASA duyurusu
metin = await kap_metin_cek(1590969)
print(metin[:1000])

Bildirim Sorguları
Şirketler
Fonlar
KAP Hakkında
TR
EN
Tüm Kategoriler
SASA POLYESTER SANAYİ A.Ş.
SASA
Kapat
Gönderim Tarihi
10.04.202618:17:18
Bildirim Tipi
ÖDA
Yıl
--
Periyot
-
Bağımsız Denetim Kuruluşunun Belirlenmesi
A+
A-
Özet Bilgi
Bağımsız Denetim Kuruluşu Seçimi
İlgili Şirketler
	
[]


İlgili Fonlar
	
[]




oda_DeterminationOfIndependentAuditCompanyAbstract|
		
Bağımsız Denetim Kuruluşunun Belirlenmesi
			
	

oda_UpdateAnnouncementFlag|
		
 
	
Yapılan Açıklama Güncelleme mi?
			
	
Hayır (No)


oda_CorrectionAnnouncementFlag|
		
 
	
Yapılan Açıklama Düzeltme mi?
			
	
Hayır (No)


oda_DateOfThePreviousNotificationAboutTheSameSubject|
		
 
	
Konuya İlişkin Daha Önce Yapılan Açıklamanın Tarihi
			
	
-


oda_DelayedAnnouncementFlag|
		
 
	
Yapılan Açıklama Ertelenmiş Bir Açıklama mı?
			
	
Hayır (No)


oda_AnnouncementContentSection|
		
 
	
Bildirim İçeriği
			
	

oda_TitleOfIndependentAuditCompany|
		
 
	
 
	
Bağımsız Denetim Kuruluşunun Ünvanı
			
	
Güney Bağımsız Denetim ve Ser

In [2]:
async def kap_metin_cek(bildirim_index: int) -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(f"https://www.kap.org.tr/tr/Bildirim/{bildirim_index}")
        await page.wait_for_timeout(2000)
        
        # Sadece duyuru içerik alanını al
        try:
            icerik = await page.inner_text(".disclosure-detail")
        except:
            try:
                icerik = await page.inner_text("main")
            except:
                icerik = await page.inner_text("body")
        
        await browser.close()
        return icerik.strip()

metin = await kap_metin_cek(1590969)
print(metin[:1000])

Tüm Kategoriler
SASA POLYESTER SANAYİ A.Ş.
SASA
Kapat
Gönderim Tarihi
10.04.202618:17:18
Bildirim Tipi
ÖDA
Yıl
--
Periyot
-
Bağımsız Denetim Kuruluşunun Belirlenmesi
A+
A-
Özet Bilgi
Bağımsız Denetim Kuruluşu Seçimi
İlgili Şirketler
	
[]


İlgili Fonlar
	
[]




oda_DeterminationOfIndependentAuditCompanyAbstract|
		
Bağımsız Denetim Kuruluşunun Belirlenmesi
			
	

oda_UpdateAnnouncementFlag|
		
 
	
Yapılan Açıklama Güncelleme mi?
			
	
Hayır (No)


oda_CorrectionAnnouncementFlag|
		
 
	
Yapılan Açıklama Düzeltme mi?
			
	
Hayır (No)


oda_DateOfThePreviousNotificationAboutTheSameSubject|
		
 
	
Konuya İlişkin Daha Önce Yapılan Açıklamanın Tarihi
			
	
-


oda_DelayedAnnouncementFlag|
		
 
	
Yapılan Açıklama Ertelenmiş Bir Açıklama mı?
			
	
Hayır (No)


oda_AnnouncementContentSection|
		
 
	
Bildirim İçeriği
			
	

oda_TitleOfIndependentAuditCompany|
		
 
	
 
	
Bağımsız Denetim Kuruluşunun Ünvanı
			
	
Güney Bağımsız Denetim ve Serbest Muhasebeci Mali Müşavirlik A.Ş.


oda_AuditPeriod|

In [13]:
async def kap_metin_cek(bildirim_index: int) -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        try:
            await page.goto(
                f"https://www.kap.org.tr/tr/Bildirim/{bildirim_index}",
                timeout=60000,
                wait_until="domcontentloaded"  # "load" yerine daha hızlı
            )
            await page.wait_for_timeout(3000)
            
            icerik = await page.inner_text("body")
            
            if "Kapat" in icerik:
                icerik = icerik.split("Kapat", 1)[1]
            
            satirlar = [s.strip() for s in icerik.split("\n") if s.strip()]
            icerik = "\n".join(satirlar)
            
        except Exception as e:
            icerik = None
        finally:
            await browser.close()
        
        return icerik

In [14]:
import time

baslangic = time.time()
metin = await kap_metin_cek(1590969)
sure = time.time() - baslangic
print(f"Süre: {sure:.1f} saniye")
print(f"4973 duyuru için tahmini: {4973 * sure / 3600:.1f} saat")

Süre: 4.4 saniye
4973 duyuru için tahmini: 6.1 saat


In [9]:
import asyncio
from pathlib import Path
import pandas as pd
import pandas as pd
from pathlib import Path

sonuc = pd.read_parquet("../data/raw/kap/announcements.parquet")
print(f"Yüklendi: {len(sonuc)} duyuru")

async def kap_metin_cek_batch(indeksler: list, eszamanli: int = 5) -> dict:
    """
    Birden fazla bildirimi eş zamanlı çek.
    eszamanli: aynı anda kaç istek atılacak
    """
    sonuclar = {}
    
    async def tek_cek(index):
        try:
            metin = await kap_metin_cek(index)
            sonuclar[index] = metin
        except Exception as e:
            sonuclar[index] = None
            print(f"❌ {index}: {e}")
    
    # Gruplara böl
    for i in range(0, len(indeksler), eszamanli):
        grup = indeksler[i:i+eszamanli]
        await asyncio.gather(*[tek_cek(idx) for idx in grup])
        print(f"✅ {min(i+eszamanli, len(indeksler))}/{len(indeksler)}")
        await asyncio.sleep(1)  # sunucuya saygılı ol
    
    return sonuclar

# Test: ilk 10 duyuru
test_indeksler = sonuc["link"].str.extract(r"/(\d+)$")[0].astype(int).tolist()[:10]
print(f"Test indeksleri: {test_indeksler[:3]}...")

baslangic = time.time()
test_sonuc = await kap_metin_cek_batch(test_indeksler, eszamanli=5)
sure = time.time() - baslangic

print(f"\n10 duyuru: {sure:.1f} saniye")
print(f"4973 duyuru için tahmini: {4973 * (sure/10) / 3600:.1f} saat")

Yüklendi: 4973 duyuru
Test indeksleri: [1434415, 1434415, 1434411]...
✅ 5/10
✅ 10/10

10 duyuru: 12.6 saniye
4973 duyuru için tahmini: 1.7 saat


In [15]:
indeksler = (
    sonuc["link"]
    .str.extract(r"/(\d+)$")[0]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
print(f"Toplam unique bildirim: {len(indeksler)}")
print(f"Tahmini süre: {len(indeksler) * 1.26 / 3600:.1f} saat")

Toplam unique bildirim: 4717
Tahmini süre: 1.7 saat


In [16]:
# Sadece ODA (Özel Durum Açıklaması) duyuruları
oda_sonuc = sonuc[sonuc["sinif"] == "ODA"]
print(f"ODA duyuru sayısı: {len(oda_sonuc)}")

oda_indeksler = (
    oda_sonuc["link"]
    .str.extract(r"/(\d+)$")[0]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
print(f"Unique ODA bildirim: {len(oda_indeksler)}")
print(f"Tahmini süre: {len(oda_indeksler) * 4.4 / 3600:.1f} saat")

ODA duyuru sayısı: 2727
Unique ODA bildirim: 2703
Tahmini süre: 3.3 saat


In [18]:
import json

CHECKPOINT_FILE = Path("../data/raw/kap/metin_checkpoint.json")
OUTPUT_FILE = Path("../data/raw/kap/metin_icerikler.json")

async def tum_metinleri_cek(indeksler: list, eszamanli: int = 5):
    # Daha önce çekilenleri yükle
    if OUTPUT_FILE.exists():
        with open(OUTPUT_FILE) as f:
            sonuclar = json.load(f)
        print(f"Checkpoint: {len(sonuclar)} duyuru zaten çekilmiş")
    else:
        sonuclar = {}
    
    # Çekilmemiş olanları bul
    kalan = [idx for idx in indeksler if str(idx) not in sonuclar]
    print(f"Kalan: {len(kalan)} duyuru")
    
    for i in range(0, len(kalan), eszamanli):
        grup = kalan[i:i+eszamanli]
        
        async def tek_cek(index):
            try:
                metin = await kap_metin_cek(index)
                sonuclar[str(index)] = metin
            except Exception as e:
                sonuclar[str(index)] = None
                print(f"❌ {index}: {e}")
        
        await asyncio.gather(*[tek_cek(idx) for idx in grup])
        
        # Her 50 duyuruda bir kaydet
        if i % 50 == 0:
            with open(OUTPUT_FILE, "w") as f:
                json.dump(sonuclar, f, ensure_ascii=False)
            print(f"✅ {min(i+eszamanli, len(kalan))}/{len(kalan)} — kaydedildi")
        else:
            print(f"✅ {min(i+eszamanli, len(kalan))}/{len(kalan)}")
        
        await asyncio.sleep(3)
    
    # Son kayıt
    with open(OUTPUT_FILE, "w") as f:
        json.dump(sonuclar, f, ensure_ascii=False)
    
    print(f"\nTamamlandı: {len(sonuclar)} duyuru")
    return sonuclar

# Çalıştır
await tum_metinleri_cek(indeksler, eszamanli=2)

Checkpoint: 157 duyuru zaten çekilmiş
Kalan: 4560 duyuru
✅ 2/4560 — kaydedildi
✅ 4/4560
✅ 6/4560
✅ 8/4560
✅ 10/4560
✅ 12/4560
✅ 14/4560
✅ 16/4560
✅ 18/4560
✅ 20/4560
✅ 22/4560
✅ 24/4560
✅ 26/4560
✅ 28/4560
✅ 30/4560
✅ 32/4560
✅ 34/4560
✅ 36/4560
✅ 38/4560
✅ 40/4560
✅ 42/4560
✅ 44/4560
✅ 46/4560
✅ 48/4560
✅ 50/4560
✅ 52/4560 — kaydedildi
✅ 54/4560
✅ 56/4560
✅ 58/4560
✅ 60/4560
✅ 62/4560
✅ 64/4560
✅ 66/4560
✅ 68/4560
✅ 70/4560
✅ 72/4560
✅ 74/4560
✅ 76/4560
✅ 78/4560
✅ 80/4560
✅ 82/4560
✅ 84/4560
✅ 86/4560
✅ 88/4560
✅ 90/4560
✅ 92/4560
✅ 94/4560
✅ 96/4560
✅ 98/4560
✅ 100/4560
✅ 102/4560 — kaydedildi
✅ 104/4560
✅ 106/4560
✅ 108/4560
✅ 110/4560
✅ 112/4560
✅ 114/4560
✅ 116/4560
✅ 118/4560
✅ 120/4560
✅ 122/4560
✅ 124/4560
✅ 126/4560
✅ 128/4560
✅ 130/4560
✅ 132/4560
✅ 134/4560
✅ 136/4560
✅ 138/4560
✅ 140/4560
✅ 142/4560
✅ 144/4560
✅ 146/4560
✅ 148/4560
✅ 150/4560
✅ 152/4560 — kaydedildi
✅ 154/4560
✅ 156/4560
✅ 158/4560
✅ 160/4560
✅ 162/4560
✅ 164/4560
✅ 166/4560
✅ 168/4560
✅ 170/4560
✅ 172/456

{'1434354': 'Gönderim Tarihi\n06.05.202519:11:24\nBildirim Tipi\nFR\nYıl\n2025\nPeriyot\n3 Aylık\nSorumluluk Beyanı (Konsolide)\nA+\nA-\nÖzet Bilgi\nKonsolide Sorumluluk Beyanı\nFinansal Tablo Niteliği\tKonsolide\nİlgili Şirketler\n[]\nİlgili Fonlar\n[]\noda_RepresentationLetterAbstract|\nSorumluluk Beyanı\noda_CorrectionAnnouncementFlag|\nYapılan Açıklama Düzeltme mi?\nHayır (No)\noda_DateOfThePreviousNotificationAboutTheSameSubject|\nKonuya İlişkin Daha Önce Yapılan Açıklamanın Tarihi\n-\noda_BoardDecisionDateAndNumberForApprovalOfFinancialStatementsAndOperatingReviewReportAbstract|http://www.xbrl.org/2003/role/verboseLabel\nFinansal Tablo ve Faaliyet Raporunun Kabulüne İlişkin Yönetim Kurulu Karar Tarihi ve Sayısı\noda_BoardDecisionDate|\nYönetim Kurulu Karar Tarihi\n06/05/2025\noda_BoardDecisionNumber|\nKarar Sayısı\n47723\noda_RepresentationLetterPreparedAccordingToRelatedCMBCommuniqueAbstract|http://www.xbrl.org/2003/role/terseLabel\nSermaye Piyasası Kurulu\'nun "Sermaye Piyasası